# Imports

In [ ]:
import copy
import os
import sys
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import IMAGENET_MEAN, IMAGENET_STD
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import label_band
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate_predictions, load_baseline_model, load_val_sets,
                                      lraspp_outputs, results_table, run_model)
from uavloc.segmentation.latency import model_latency, refiner_latency, standard_tile
from uavloc.segmentation.refine.segfix import (DIRECTION_IGNORE, BoundaryField, SegFixConfig, SegFixNet,
                                               SegFixRefiner, batch_targets, boundary_targets, field_scores,
                                               predict_field, refine_with_field, segfix_loss)
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs  # the LR-ASPP training samplers and holdout ban; PoC-local, imported not copied

# Params

In [ ]:
EXP = "014-boundary-displacement"
NOTE = ""
LOG_RUN = True
LOG_BASELINE = False
BOUNDARY_WIDTH_PX = 4
OFFSETS_PX = [2, 4, 8]
THRESHOLD = 0.5
NET_WIDTHS = [16, 32, 64]
NET_DILATIONS = [2, 4]
SEED = 42
DEVICE = "mps"
CASES_DEVICE = "cpu"
STEPS = 2500
BATCH = 8
CROP_PX = 512
LR = 2e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
RARE_CROP_P = 0.3
GAP_PX = 64
LOG_EVERY = 250
BOUNDARY_PX = 4
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 5
LATENCY_RUNS = 30
LATENCY_BUDGET_RATIO = 0.35
TARGET_DEMO_SAMPLES = 3
ZOOMS = [["birky-strip-google", 0, 120, 1400, 376, 1912], ["bushcha", 0, 1664, 640, 1920, 1152]]

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
VILLAGES = DERIVED / "ua-village-annotated"
EXP_DIR = experiment_dir(EXP)
ARTIFACTS, FIGURES = EXP_DIR / "artifacts", EXP_DIR / "figures"
CHECKPOINT = ARTIFACTS / f"segfix-k{BOUNDARY_WIDTH_PX}-seed{SEED}.pt"
set_seed(SEED)
rng = np.random.default_rng(SEED)

In [ ]:
sat = vs.load_satellite(VILLAGES, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
assert all(s.loc != HOLDOUT_LOCATION for s in sat_train)
assert all(s.banned[:STRIP_ROWS].all() for s in sat_train if s.loc == STRIP_LOCATION)
sampler = vs.SatelliteSampler(sat_train, CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets()
{name: len(items) for name, items in val_sets.items()}

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
def direction_image(direction: np.ndarray) -> np.ndarray:
    """Direction bins in a cyclic colour map, undefined pixels white."""
    out = (plt.cm.hsv(np.clip(direction, 0, None) / 8)[..., :3] * 255).astype(np.uint8)
    out[direction == DIRECTION_IGNORE] = 255
    return out

In [ ]:
demo_rng = np.random.default_rng(SEED)
fig_targets, axes = plt.subplots(TARGET_DEMO_SAMPLES, 4, figsize=(14, 3.6 * TARGET_DEMO_SAMPLES))
for row in axes:
    x, y = sampler(demo_rng)
    boundary, direction = boundary_targets(y, BOUNDARY_WIDTH_PX)
    rgb = np.clip(x.transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
    for ax, img, title in zip(row, [rgb, colour(y), boundary, direction_image(np.where(boundary, direction, DIRECTION_IGNORE))],
                              ["RGB (augmented)", "label", f"boundary target, k = {BOUNDARY_WIDTH_PX}", "direction bin (hsv)"]):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); fig_targets.savefig(FIGURES / f"targets-k{BOUNDARY_WIDTH_PX}.png", dpi=80, bbox_inches="tight")

# Model

In [ ]:
lraspp = load_baseline_model(CASES_DEVICE)
cases = run_model(lraspp, val_sets, CASES_DEVICE)

In [ ]:
net = SegFixNet(SegFixConfig(tuple(NET_WIDTHS), tuple(NET_DILATIONS))).to(DEVICE)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=WARMUP_FRAC)
n_params = sum(p.numel() for p in net.parameters())
n_params

# Run

In [ ]:
def train_step(x: torch.Tensor, y: torch.Tensor) -> tuple[float, float]:
    boundary, direction = batch_targets(y, BOUNDARY_WIDTH_PX)
    bce, ce = segfix_loss(*net(x.to(DEVICE)), boundary.to(DEVICE), direction.to(DEVICE))
    opt.zero_grad(); (bce + ce).backward(); opt.step(); sched.step()
    return bce.item(), ce.item()

In [ ]:
history, window, t0 = [], [], time.time()
net.train()
for step in range(1, STEPS + 1):
    window.append(train_step(*vs.batch_of(sampler, rng, BATCH)))
    if step % LOG_EVERY == 0:
        bce, ce = np.mean(window, axis=0)
        history.append({"step": step, "bce": bce, "ce": ce, "minutes": (time.time() - t0) / 60})
        print(f"step {step}  bce {bce:.4f}  ce {ce:.4f}  [{history[-1]['minutes']:.1f} min]", flush=True)
        window = []
train_minutes = (time.time() - t0) / 60
torch.save(net.state_dict(), CHECKPOINT)

In [ ]:
fields = {name: [predict_field(net, c.image.rgb, DEVICE) for c in items] for name, items in cases.items()}
base_preds = {name: [bilinear_argmax(c.output, c.image.rgb) for c in items] for name, items in cases.items()}

In [ ]:
def oracle_field(label: np.ndarray) -> BoundaryField:
    """GT boundary and direction; boundary pixels without a defined direction stay put."""
    boundary, direction = boundary_targets(label, BOUNDARY_WIDTH_PX)
    moved = boundary & (direction != DIRECTION_IGNORE)
    return BoundaryField(moved.astype(np.float32), np.clip(direction, 0, None).astype(np.uint8))

In [ ]:
oracle_fields = {name: [oracle_field(c.image.label) for c in items] for name, items in cases.items()}

In [ ]:
def displaced(field_sets: dict[str, list[BoundaryField]], offset_px: int) -> dict[str, list[np.ndarray]]:
    return {name: [refine_with_field(p, f, offset_px, THRESHOLD) for p, f in zip(base_preds[name], field_sets[name])]
            for name in base_preds}

In [ ]:
assert all((a == b).all() for n in base_preds for a, b in zip(base_preds[n], displaced(oracle_fields, 0)[n]))
baseline = evaluate_predictions(base_preds, cases, BOUNDARY_PX)
learned = {s: evaluate_predictions(displaced(fields, s), cases, BOUNDARY_PX) for s in OFFSETS_PX}
oracle = {s: evaluate_predictions(displaced(oracle_fields, s), cases, BOUNDARY_PX) for s in OFFSETS_PX}

In [ ]:
summary = pd.concat({"baseline": results_table(baseline),
                     **{f"segfix s={s}": results_table(r) for s, r in learned.items()},
                     **{f"oracle s={s}": results_table(r) for s, r in oracle.items()}}, names=["run", "set"])
summary.round(4)

In [ ]:
per_class = pd.DataFrame({"baseline": {n: baseline[n]["biou_per_class"] for n in VAL_SETS},
                          **{f"segfix s={s}": {n: r[n]["biou_per_class"] for n in VAL_SETS} for s, r in learned.items()}})
per_class = pd.concat({col: pd.DataFrame(per_class[col].to_dict()).T for col in per_class}, names=["run", "set"])
per_class.round(3)

In [ ]:
diagnostics = pd.DataFrame({n: pd.DataFrame([field_scores(f, c.image.label, BOUNDARY_WIDTH_PX, THRESHOLD)
                                             for f, c in zip(fields[n], cases[n])]).mean() for n in VAL_SETS}).T
diagnostics.loc[MEAN_KEY] = diagnostics.mean()
diagnostics.round(3)

In [ ]:
latency_case = cases[LATENCY_SET][0]
latency_tile = standard_tile(latency_case.image.rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(lraspp, latency_tile, CASES_DEVICE)
cpu_net = copy.deepcopy(net).cpu()
timing = dict(warmup=LATENCY_WARMUP, runs=LATENCY_RUNS, threads=LATENCY_THREADS)
latency = {"model": model_latency(lraspp, latency_tile, **timing),
           "argmax": refiner_latency(bilinear_argmax, latency_output, latency_tile, **timing),
           **{f"segfix s={s}": refiner_latency(SegFixRefiner(cpu_net, "cpu", s, THRESHOLD), latency_output,
                                               latency_tile, **timing) for s in OFFSETS_PX}}
latency_table = pd.DataFrame({k: vars(v) for k, v in latency.items()}).T
latency_table["ratio_to_model"] = latency_table["median_ms"] / latency["model"].median_ms
latency_table.round(3)

# Plots

In [ ]:
fig_curve, ax = plt.subplots(figsize=(6, 3.5))
hist = pd.DataFrame(history)
ax.plot(hist["step"], hist["bce"], "-o", label="boundary BCE"); ax.plot(hist["step"], hist["ce"], "-s", label="direction CE")
ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.grid(alpha=.3); ax.legend()
fig_curve.savefig(FIGURES / f"curve-k{BOUNDARY_WIDTH_PX}-seed{SEED}.png", dpi=100, bbox_inches="tight")

In [ ]:
def band_overlay(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Predicted classes, with the label's 1-px class contours drawn in white."""
    out = colour(pred)
    out[np.any([label_band(label, c, 1) for c in range(len(CLASSES))], axis=0)] = 255
    return out

In [ ]:
best_s = max(OFFSETS_PX, key=lambda s: learned[s][MEAN_KEY]["biou"])
fig_zoom, axes = plt.subplots(len(ZOOMS), 5, figsize=(26, 3.3 * len(ZOOMS)), squeeze=False)
for row, (name, k, top, left, bottom, right) in zip(axes, ZOOMS):
    case, box = cases[name][k], np.s_[top:bottom, left:right]
    lab = case.image.label[box]
    ours = refine_with_field(base_preds[name][k], fields[name][k], best_s, THRESHOLD)
    orc = refine_with_field(base_preds[name][k], oracle_fields[name][k], best_s)
    panels = [(case.image.rgb[box], f"{name} RGB"), (colour(lab), "label"),
              (band_overlay(base_preds[name][k][box], lab), "LR-ASPP baseline"),
              (band_overlay(ours[box], lab), f"SegFix k={BOUNDARY_WIDTH_PX} s={best_s}"),
              (band_overlay(orc[box], lab), f"oracle k={BOUNDARY_WIDTH_PX} s={best_s}")]
    for ax, (img, title) in zip(row, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); fig_zoom.savefig(FIGURES / f"zoom-k{BOUNDARY_WIDTH_PX}-seed{SEED}.png", dpi=90, bbox_inches="tight")

In [ ]:
def scalar_metrics(results: dict, prefix: str = "") -> dict:
    out = {f"{prefix}{k}_mean": v for k, v in results[MEAN_KEY].items()}
    out.update({f"{prefix}biou_{n}": results[n]["biou"] for n in VAL_SETS})
    out.update({f"{prefix}miou_{n}": results[n]["miou"] for n in VAL_SETS})
    return out

In [ ]:
base_config = {"method": "segfix", "base_model": "lraspp-bn-satellite bilinear argmax", "boundary_px": BOUNDARY_PX,
               "boundary_width_px": BOUNDARY_WIDTH_PX, "threshold": THRESHOLD, "net_widths": NET_WIDTHS,
               "net_dilations": NET_DILATIONS, "steps": STEPS, "batch": BATCH, "crop_px": CROP_PX, "lr": LR,
               "weight_decay": WEIGHT_DECAY, "rare_crop_p": RARE_CROP_P, "input": "rgb",
               "train_images": [f"{s.loc}/{s.date}" for s in sat_train]}
train_metrics = {"train_minutes": train_minutes, "final_bce": history[-1]["bce"], "final_ce": history[-1]["ce"],
                 "n_params": n_params, "model_ms": latency["model"].median_ms, **diagnostics.loc[MEAN_KEY].to_dict()}

In [ ]:
logged = []
if LOG_RUN and LOG_BASELINE:
    logged.append(log_run(exp=EXP, config={"method": "baseline", "base_model": base_config["base_model"],
                                           "boundary_px": BOUNDARY_PX},
                          metrics={**scalar_metrics(baseline), "refiner_ms": latency["argmax"].median_ms,
                                   "model_ms": latency["model"].median_ms},
                          note="LR-ASPP bilinear argmax through this notebook", seed=SEED, device=CASES_DEVICE)["run"])
for s in OFFSETS_PX if LOG_RUN else []:
    lat = latency[f"segfix s={s}"]
    metrics = {**scalar_metrics(learned[s]), **scalar_metrics(oracle[s], "oracle_"), **train_metrics,
               "delta_biou_mean": learned[s][MEAN_KEY]["biou"] - baseline[MEAN_KEY]["biou"],
               "delta_miou_mean": learned[s][MEAN_KEY]["miou"] - baseline[MEAN_KEY]["miou"],
               "refiner_ms": lat.median_ms, "refiner_p90_ms": lat.p90_ms,
               "latency_ratio": lat.median_ms / latency["model"].median_ms, "loss_history": history}
    logged.append(log_run(exp=EXP, config={**base_config, "offset_px": s}, metrics=metrics, note=NOTE or f"k={BOUNDARY_WIDTH_PX} s={s}",
                          figures=[fig_zoom, fig_curve], seed=SEED, device=DEVICE,
                          checkpoint=str(CHECKPOINT.relative_to(REPO_ROOT)))["run"])
logged